# 🌲 Random Forest Interactive Explorer
### Breast Cancer Dataset — scikit-learn

Use the interactive controls below to explore how a **Random Forest** classifier learns.

| View | What it shows |
|------|--------------|
| **Accuracy** | Train vs test accuracy as tree count grows |
| **Loss** | Log-loss as tree count grows |
| **Accuracy + Loss** | Both curves side by side |
| **Single Tree** | One individual decision tree from inside the forest |

---

In [1]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import ipywidgets as widgets
from ipywidgets import interact, interactive_output, HBox, VBox, Label
from IPython.display import display, clear_output

from sklearn.datasets import load_breast_cancer
from sklearn.ensemble import RandomForestClassifier
from sklearn.tree import plot_tree
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, log_loss

%matplotlib inline
plt.rcParams.update({
    "figure.facecolor": "#f9f9f9",
    "axes.facecolor":   "#ebebeb",
    "axes.grid":        True,
    "grid.color":       "white",
    "grid.linewidth":   1.3,
    "font.size":        11,
})

print("✅ All libraries loaded.")

✅ All libraries loaded.


In [2]:
cancer = load_breast_cancer()
X, y   = cancer.data, cancer.target

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=42, stratify=y
)

print(f"Dataset  : {len(X)} samples, {X.shape[1]} features")
print(f"Training : {len(X_train)} samples")
print(f"Testing  : {len(X_test)} samples")
print(f"Classes  : {', '.join(cancer.target_names)}")

Dataset  : 569 samples, 30 features
Training : 426 samples
Testing  : 143 samples
Classes  : malignant, benign


In [3]:
# Pre-compute metrics for 1 → 200 trees (step of 2).
# This takes ~15 seconds but makes all sliders feel instant.

TREE_COUNTS  = list(range(1, 202, 2))
train_accs, test_accs     = [], []
train_losses, test_losses = [], []
forests = {}   # store every trained forest

print("Training forests — please wait (~15 sec)...")

for i, n in enumerate(TREE_COUNTS):
    rf = RandomForestClassifier(n_estimators=n, random_state=42, n_jobs=-1)
    rf.fit(X_train, y_train)
    forests[n] = rf
    train_accs.append(  accuracy_score(y_train, rf.predict(X_train)) * 100)
    test_accs.append(   accuracy_score(y_test,  rf.predict(X_test))  * 100)
    train_losses.append(log_loss(y_train, rf.predict_proba(X_train)))
    test_losses.append( log_loss(y_test,  rf.predict_proba(X_test)))
    if (i + 1) % 25 == 0:
        print(f"  {i+1}/{len(TREE_COUNTS)} complete...")

# Feature importances from a 100-tree forest
rf100       = forests[101]
importances = rf100.feature_importances_
sorted_idx  = np.argsort(importances)[::-1][:10]
top_feats   = [cancer.feature_names[i] for i in sorted_idx]
top_imps    = importances[sorted_idx]

print("\n✅ All forests ready!")

Training forests — please wait (~15 sec)...
  25/101 complete...
  50/101 complete...
  75/101 complete...
  100/101 complete...

✅ All forests ready!


In [4]:
BLUE   = '#1565C0'
ORANGE = '#E65100'
GREEN  = '#2E7D32'

def nearest_idx(n):
    return min(range(len(TREE_COUNTS)), key=lambda i: abs(TREE_COUNTS[i] - n))

def style_ax(ax, xlabel, ylabel, title):
    ax.set_xlabel(xlabel, fontsize=10)
    ax.set_ylabel(ylabel, fontsize=10)
    ax.set_title(title, fontsize=12, fontweight='bold', pad=10)
    ax.spines[['top','right']].set_visible(False)

def draw_accuracy(ax, n_trees):
    idx = nearest_idx(n_trees)
    n   = TREE_COUNTS[idx]
    ax.plot(TREE_COUNTS, train_accs, color=BLUE,   lw=2, label='Training accuracy')
    ax.plot(TREE_COUNTS, test_accs,  color=ORANGE, lw=2, label='Test accuracy')
    ax.fill_between(TREE_COUNTS, train_accs, test_accs, alpha=0.07, color='gray')
    ax.axvline(n, color=GREEN, ls='--', lw=2, label=f'Selected: {n} trees')
    ax.scatter([n], [train_accs[idx]], color=BLUE,   s=80, zorder=5)
    ax.scatter([n], [test_accs[idx]],  color=ORANGE, s=80, zorder=5)
    ax.set_ylim(87, 102)
    gap = train_accs[idx] - test_accs[idx]
    ax.annotate(
        f'Train: {train_accs[idx]:.1f}%\nTest:  {test_accs[idx]:.1f}%\nGap:   {gap:.1f}%',
        xy=(n, test_accs[idx]),
        xytext=(min(n + 18, 165), 89.5),
        fontsize=9, color='#333',
        arrowprops=dict(arrowstyle='->', color=GREEN, lw=1.2),
        bbox=dict(boxstyle='round,pad=0.4', fc='white', ec=GREEN, alpha=0.95)
    )
    style_ax(ax, 'Number of trees in the forest', 'Accuracy (%)',
             'Accuracy vs Number of Trees')
    ax.legend(fontsize=9, loc='lower right')

def draw_loss(ax, n_trees):
    idx = nearest_idx(n_trees)
    n   = TREE_COUNTS[idx]
    ax.plot(TREE_COUNTS, train_losses, color=BLUE,   lw=2, label='Training loss')
    ax.plot(TREE_COUNTS, test_losses,  color=ORANGE, lw=2, label='Test loss')
    ax.axvline(n, color=GREEN, ls='--', lw=2, label=f'Selected: {n} trees')
    ax.scatter([n], [train_losses[idx]], color=BLUE,   s=80, zorder=5)
    ax.scatter([n], [test_losses[idx]],  color=ORANGE, s=80, zorder=5)
    ax.annotate(
        f'Test loss: {test_losses[idx]:.4f}',
        xy=(n, test_losses[idx]),
        xytext=(min(n + 18, 155), test_losses[idx] + 0.05),
        fontsize=9, color='#333',
        arrowprops=dict(arrowstyle='->', color=GREEN, lw=1.2),
        bbox=dict(boxstyle='round,pad=0.4', fc='white', ec=GREEN, alpha=0.95)
    )
    style_ax(ax, 'Number of trees in the forest', 'Log-loss  (lower = better)',
             'Loss vs Number of Trees')
    ax.legend(fontsize=9, loc='upper right')

def draw_features(ax):
    colors = [BLUE if i == 0 else '#78909C' for i in range(10)]
    y_pos  = np.arange(10)
    ax.barh(y_pos, top_imps[::-1], color=colors[::-1], edgecolor='white', height=0.65)
    ax.set_yticks(y_pos)
    ax.set_yticklabels([top_feats[9 - i] for i in range(10)], fontsize=9)
    for i, v in enumerate(top_imps[::-1]):
        ax.text(v + 0.0005, i, f'{v*100:.1f}%', va='center', fontsize=8.5)
    style_ax(ax, 'Importance score', '', 'Top 10 Most Important Features')

print("✅ Helper functions defined.")

✅ Helper functions defined.


In [6]:
# ── Widgets ───────────────────────────────────────────────
view_toggle = widgets.ToggleButtons(
    options=['Accuracy', 'Loss', 'Accuracy + Loss', 'Single Tree'],
    description='',
    button_style='',
    style={'button_width': '140px', 'description_width': '0px'},
    layout=widgets.Layout(margin='0 0 12px 0')
)

tree_slider = widgets.IntSlider(
    value=101, min=1, max=201, step=2,
    description='Trees:',
    continuous_update=True,
    style={'description_width': '60px'},
    layout=widgets.Layout(width='550px')
)

tree_index_slider = widgets.IntSlider(
    value=1, min=1, max=101, step=1,
    description='Tree #:',
    continuous_update=True,
    style={'description_width': '60px'},
    layout=widgets.Layout(width='400px')
)
tree_index_label = widgets.Label(
    value='← Pick which individual tree inside the forest to inspect',
    layout=widgets.Layout(margin='6px 0 0 8px')
)

tree_index_row = HBox(
    [tree_index_slider, tree_index_label],
    layout=widgets.Layout(display='none')   # hidden until Single Tree view
)

out = widgets.Output()

# ── Update function ───────────────────────────────────────
def update(change=None):
    view    = view_toggle.value
    n_trees = tree_slider.value
    t_idx   = tree_index_slider.value - 1   # 0-based

    # Show / hide the tree-index row
    tree_index_row.layout.display = '' if view == 'Single Tree' else 'none'

    # Clamp tree index to trees available in this forest
    n_actual = TREE_COUNTS[nearest_idx(n_trees)]
    max_t    = len(forests[n_actual].estimators_)
    if tree_index_slider.max != max_t:
        tree_index_slider.max = max_t
    if t_idx >= max_t:
        t_idx = max_t - 1

    with out:
        clear_output(wait=True)

        if view == 'Single Tree':
            # ── Single tree view ──────────────────────────
            fig, ax = plt.subplots(figsize=(16, 7))
            fig.patch.set_facecolor('#f9f9f9')
            rf   = forests[n_actual]
            tree = rf.estimators_[t_idx]

            plot_tree(
                tree,
                feature_names=list(cancer.feature_names),
                class_names=list(cancer.target_names),
                filled=True,
                rounded=True,
                max_depth=10,
                ax=ax,
                fontsize=8,
                impurity=False,
                precision=2,
            )
            ax.set_title(
                f'Tree #{t_idx + 1} out of {n_actual} trees in this forest'
                f'  —  showing top 3 levels of depth\n'
                f'Blue = Malignant  |  Orange = Benign  |  '
                f'Darker fill = more confident prediction',
                fontsize=11, fontweight='bold', pad=14
            )
            fig.text(
                0.5, 0.01,
                'Each box: the question asked  |  samples reaching this node  |  '
                'class the node predicts if the tree stopped here',
                ha='center', fontsize=9, color='#666',
                bbox=dict(boxstyle='round,pad=0.4', fc='white', ec='#ccc', alpha=0.9)
            )
            plt.tight_layout(rect=[0, 0.04, 1, 1])
            plt.show()
            plt.close(fig)

        elif view == 'Accuracy':
            fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))
            fig.patch.set_facecolor('#f9f9f9')
            draw_accuracy(ax1, n_trees)
            draw_features(ax2)
            plt.tight_layout()
            plt.show()
            plt.close(fig)

        elif view == 'Loss':
            fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))
            fig.patch.set_facecolor('#f9f9f9')
            draw_loss(ax1, n_trees)
            draw_features(ax2)
            plt.tight_layout()
            plt.show()
            plt.close(fig)

        else:   # Accuracy + Loss
            fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))
            fig.patch.set_facecolor('#f9f9f9')
            draw_accuracy(ax1, n_trees)
            draw_loss(ax2, n_trees)
            plt.tight_layout()
            plt.show()
            plt.close(fig)

# Attach observers
view_toggle.observe(update, names='value')
tree_slider.observe(update, names='value')
tree_index_slider.observe(update, names='value')

# ── Layout & display ──────────────────────────────────────
header = widgets.HTML(
    value='<b style="font-size:14px">Random Forest Explorer</b>'
          '<span style="color:#888; margin-left:12px; font-size:12px">'
          'Breast Cancer Dataset</span>'
)

ui = VBox([
    header,
    view_toggle,
    HBox([widgets.Label('Number of trees:', layout=widgets.Layout(width='120px',
                        margin='6px 0 0 0')), tree_slider]),
    tree_index_row,
    out
])

display(ui)
update()   # draw initial state

---
## Key Takeaways

1. **More trees = more stable** — accuracy and loss both improve quickly at first, then level off around 50–100 trees.
2. **Diminishing returns** — adding tree #200 barely helps compared to going from tree #1 to #10.
3. **Single Tree view** — flip through individual trees to see how different each "voter" looks. Some are very different from each other — that diversity is *why* the ensemble works.
4. **"Worst area"** is the most important feature — the largest cell nucleus area in a biopsy is the single strongest predictor of malignancy.
5. **Real data** — this dataset is used in actual clinical decision-support research.